# Restaurant Annual Turnover Prediction

**Goal:** predict Annual Turnover (INR) for 500 test restaurants and minimize RMSE.

**Student register number:** submission

**Student name:** submission

**Team name:** submission

This notebook includes data checks, evidence-based inferences, five-fold model validation, final training, and a checked submission CSV.

### Run in Google Colab
1. Upload this notebook to Colab.
2. Run the cells from top to bottom.
3. When prompted, upload Train_dataset.csv, Test_dataset.csv, Data_Dictionary.csv, and Sample_Solution.csv.
4. Download submission.csv from the final cell.

The leaderboard screenshot provided with the task showed a leading RMSE of about INR 11.95 million. This is a reference point, not a promised score. Use cross-validation to select a model; the test target is not available.

## 1. Imports and setup

If CatBoost is missing in your Colab runtime, uncomment and run the install line below once. CatBoost is used for a mixed dataset of numeric and categorical restaurant features.

In [ ]:
import importlib.util
import subprocess
import sys
import warnings
from pathlib import Path

# Install only if Colab or the local notebook environment is missing a required package.
missing = [name for name in ["catboost", "sklearn"] if importlib.util.find_spec(name) is None]
if missing:
    print("Installing required modeling packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost", "scikit-learn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Random seed:", RANDOM_STATE)

## 2. Load and verify the files

The train file contains the target; the test file does not. Registration Number is an identifier, not a predictor. The train and test files use different spellings for the endorsement column, so the code aligns that name before modeling.

In [ ]:
try:
    from google.colab import files
    colab_files_available = True
except ImportError:
    files = None
    colab_files_available = False

DATA_DIR = Path("/content")
required_files = [
    "Train_dataset.csv",
    "Test_dataset.csv",
    "Data_Dictionary.csv",
    "Sample_Solution.csv",
]

if not all((DATA_DIR / name).exists() for name in required_files):
    local_downloads = Path("~/Downloads")
    if all((local_downloads / name).exists() for name in required_files):
        DATA_DIR = local_downloads
    elif colab_files_available:
        print("Upload all four CSV files when the file picker opens.")
        files.upload()
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(
            "Place the four CSV files in /content (Colab) or set DATA_DIR to their folder."
        )

train = pd.read_csv(DATA_DIR / "Train_dataset.csv")
test = pd.read_csv(DATA_DIR / "Test_dataset.csv")
dictionary = pd.read_csv(DATA_DIR / "Data_Dictionary.csv")
sample = pd.read_csv(DATA_DIR / "Sample_Solution.csv")

# Standardize the spelling mismatch before comparing feature columns.
train = train.rename(columns={"Endorsed By": "Endoresed By"})

assert "Annual Turnover" in train.columns
assert "Annual Turnover" not in test.columns
assert len(test) == 500
assert list(sample.columns) == ["Registration Number", "Annual Turnover"]
assert sample["Registration Number"].equals(test["Registration Number"])

test_ids = test["Registration Number"].copy()
y = train["Annual Turnover"].astype(float)
X_raw = train.drop(columns=["Annual Turnover", "Registration Number"]).copy()
X_test_raw = test.drop(columns=["Registration Number"]).copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Feature columns aligned:", list(X_raw.columns) == list(X_test_raw.columns))
print("Training IDs:", int(train["Registration Number"].min()), "to", int(train["Registration Number"].max()))
print("Test IDs:", int(test_ids.min()), "to", int(test_ids.max()))
print("Sample solution is a template; its Annual Turnover values are all:", sample["Annual Turnover"].unique().tolist())

## 3. Explore the target and record inferences

The target is right-skewed: its mean is above its median, and its extreme high values matter under RMSE. The notebook also checks numeric rank correlations and missingness. These are descriptive patterns in this dataset, not causal claims.

In [ ]:
print("Annual Turnover summary (INR):")
print(y.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(0).to_string())
print("Unique turnover values:", y.nunique())
print("Skew:", round(float(y.skew()), 3))

numeric = train.select_dtypes(include=[np.number]).drop(
    columns=["Registration Number", "Annual Turnover"], errors="ignore"
)
rank_correlations = numeric.rank().corrwith(y.rank()).sort_values(
    key=lambda s: s.abs(), ascending=False
)
print("\nLargest numeric rank correlations with turnover:")
print(rank_correlations.head(10).round(3).to_string())

print("\nMissing values in training (%):")
print((train.isna().mean().mul(100).sort_values(ascending=False).head(12)).round(1).to_string())

print("\nTurnover by Zomato rating:")
print(
    train.groupby("Restaurant Zomato Rating")["Annual Turnover"]
    .agg(["count", "median", "mean"])
    .round(0)
    .to_string()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y / 1_000_000, bins=40, color="#2563eb", edgecolor="white")
ax.set_title("Training target distribution")
ax.set_xlabel("Annual Turnover (INR millions)")
ax.set_ylabel("Restaurant count")
fig.tight_layout()
plt.show()

### Inferences to include

- The training median is INR 30 million; the mean is about INR 30.73 million. The maximum is INR 400 million and the 99th percentile is INR 100 million, so a small high-turnover tail can dominate RMSE.
- Hygiene rating, Facebook popularity, live music rating, and Instagram popularity have some of the strongest numeric rank associations with turnover in this sample. Use them as model signals, not causal explanations.
- Median turnover generally rises across Zomato rating levels 1 through 5. Rating 5 has only 38 examples, so that group estimate is less stable.
- Several event ratings are missing for many restaurants. The data dictionary says these blanks mean the restaurant does not offer that event; keep availability information instead of treating a blank as a zero rating.
- The training IDs (60001–63493) and test IDs (20001–20500) do not overlap. Exclude IDs from modeling and use test IDs only to label predictions.

## 4. Feature preparation

Normalize category text, parse the opening date into year/month, split the two cuisines into additional fields, and add indicators for event availability. Numeric missing values are left for CatBoost to handle; categorical missing values are replaced by a text label.

In [ ]:
OPTIONAL_RATING_COLUMNS = [
    "Live Music Rating",
    "Comedy Gigs Rating",
    "Value Deals Rating",
    "Live Sports Rating",
]

def prepare_features(frame):
    data = frame.copy()

    opening_date = pd.to_datetime(
        data["Opening Day of Restaurant"], format="%d-%m-%Y", errors="coerce"
    )
    data["Opening Year"] = opening_date.dt.year.astype(float)
    data["Opening Month"] = opening_date.dt.month.astype(float)
    data = data.drop(columns=["Opening Day of Restaurant"])

    for col in OPTIONAL_RATING_COLUMNS:
        data["Offers " + col.replace(" Rating", "")] = data[col].notna().astype(int)

    cuisine = data["Cuisine"].astype("string").fillna("Missing")
    cuisine_parts = cuisine.str.split(",", n=1, expand=True)
    data["Cuisine 1"] = cuisine_parts[0].str.strip().str.lower()
    if cuisine_parts.shape[1] > 1:
        data["Cuisine 2"] = cuisine_parts[1].str.strip().str.lower()
    else:
        data["Cuisine 2"] = "Missing"

    for col in ["Cuisine", "City", "Restaurant Location", "Endoresed By", "Restaurant Type", "Restaurant Theme"]:
        data[col] = (
            data[col].astype("string")
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .fillna("Missing")
        )

    # The dataset uses -1 for some unknown-city records; retain that category and flag it.
    data["City Is Unknown"] = (data["City"] == "-1").astype(int)

    for col in data.select_dtypes(include=["object", "string"]).columns:
        data[col] = data[col].fillna("Missing").astype(str)

    return data

X = prepare_features(X_raw)
X_test = prepare_features(X_test_raw)
assert list(X.columns) == list(X_test.columns), "Train/test feature columns differ after preparation."
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
print("Prepared feature shape:", X.shape)
print("Categorical columns:", cat_cols)
print("Train/test prepared columns match:", list(X.columns) == list(X_test.columns))

## 5. Validation: mean baseline, CatBoost, and Extra Trees

Use shuffled five-fold cross-validation on training data. The out-of-fold predictions give a comparable RMSE estimate for every candidate. The log-target model is converted back to INR before scoring. Alongside standard RMSE CatBoost, test a Huber-loss model to reduce the influence of unusually large turnover values, then check the fixed blend on two additional shuffled five-fold splits. Keep a blend only when its OOF RMSE improves enough to be meaningful.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(kf.split(X))
oof_predictions = {}
best_iterations = {}
candidate_info = {}
score_rows = []

# Mean baseline: each validation fold is predicted using only that fold's training mean.
baseline_oof = np.zeros(len(y), dtype=float)
for train_idx, valid_idx in folds:
    baseline_oof[valid_idx] = y.iloc[train_idx].mean()
oof_predictions["Mean baseline"] = baseline_oof
score_rows.append({"Model": "Mean baseline", "OOF RMSE (INR)": rmse(y, baseline_oof)})

def catboost_oof(depth, log_target=False, label=None, objective="RMSE"):
    key = label or ("CatBoost log depth " if log_target else "CatBoost raw depth ") + str(depth)
    oof = np.zeros(len(y), dtype=float)
    iters = []
    for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
        y_fit = np.log1p(y.iloc[train_idx]) if log_target else y.iloc[train_idx]
        y_valid = np.log1p(y.iloc[valid_idx]) if log_target else y.iloc[valid_idx]
        model = CatBoostRegressor(
            loss_function=objective,
            eval_metric="RMSE",
            iterations=1600,
            learning_rate=0.04,
            depth=depth,
            l2_leaf_reg=5,
            random_seed=RANDOM_STATE + fold_number,
            thread_count=4,
            verbose=False,
        )
        model.fit(
            X.iloc[train_idx],
            y_fit,
            cat_features=cat_cols,
            eval_set=(X.iloc[valid_idx], y_valid),
            early_stopping_rounds=120,
            verbose=False,
        )
        pred = model.predict(X.iloc[valid_idx])
        if log_target:
            pred = np.expm1(pred)
        oof[valid_idx] = np.maximum(pred, 0)
        iters.append(int(model.best_iteration_) + 1)
        print(f"{key} | fold {fold_number}: RMSE INR {rmse(y.iloc[valid_idx], oof[valid_idx]):,.0f}, trees {iters[-1]}")
    oof_predictions[key] = oof
    best_iterations[key] = iters
    candidate_info[key] = {"kind": "catboost", "depth": depth, "log_target": log_target, "objective": objective}
    score_rows.append({"Model": key, "OOF RMSE (INR)": rmse(y, oof)})
    return key

for depth in [5, 6, 7]:
    catboost_oof(depth=depth, log_target=False)
catboost_oof(depth=6, log_target=True)
catboost_oof(
    depth=6,
    label="CatBoost Huber delta 12m",
    objective="Huber:delta=12000000",
)

# One-hot Extra Trees provides a different model family for a possible blend.
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

def make_extra_trees_pipeline():
    preprocess = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_cols),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
            ]), categorical_cols),
        ],
        remainder="drop",
    )
    model = ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline([("preprocess", preprocess), ("model", model)])

et_oof = np.zeros(len(y), dtype=float)
for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    et_model = make_extra_trees_pipeline()
    et_model.fit(X.iloc[train_idx], y.iloc[train_idx])
    et_oof[valid_idx] = np.maximum(et_model.predict(X.iloc[valid_idx]), 0)
    print(f"Extra Trees | fold {fold_number}: RMSE INR {rmse(y.iloc[valid_idx], et_oof[valid_idx]):,.0f}")
oof_predictions["Extra Trees"] = et_oof
candidate_info["Extra Trees"] = {"kind": "extra_trees"}
score_rows.append({"Model": "Extra Trees", "OOF RMSE (INR)": rmse(y, et_oof)})

results = pd.DataFrame(score_rows).sort_values("OOF RMSE (INR)").reset_index(drop=True)
print("\nOut-of-fold model comparison:")
print(results.to_string(index=False, formatters={"OOF RMSE (INR)": lambda value: f"{value:,.0f}"}))


# Stability check on two fresh shuffled five-fold assignments. The blend weight is
# fixed from the primary CV rather than re-optimized on these checks.
repeat_cv_rows = []
for split_seed in [2026, 17]:
    repeat_folds = list(KFold(n_splits=5, shuffle=True, random_state=split_seed).split(X))
    repeat_oof = {"RMSE": np.zeros(len(y)), "Huber": np.zeros(len(y))}
    for model_name, loss in [("RMSE", "RMSE"), ("Huber", "Huber:delta=12000000")]:
        for fold_number, (train_idx, valid_idx) in enumerate(repeat_folds, start=1):
            model = CatBoostRegressor(
                loss_function=loss,
                eval_metric="RMSE",
                iterations=1800,
                learning_rate=0.04,
                depth=6,
                l2_leaf_reg=5,
                random_seed=1000 + split_seed + fold_number,
                thread_count=4,
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(
                X.iloc[train_idx], y.iloc[train_idx],
                cat_features=cat_cols,
                eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
                early_stopping_rounds=120,
                verbose=False,
            )
            repeat_oof[model_name][valid_idx] = np.maximum(
                model.predict(X.iloc[valid_idx]), 0
            )
            print(f"Repeat CV {split_seed} | {model_name} | fold {fold_number} complete")
    fixed_blend = 0.56 * repeat_oof["RMSE"] + 0.44 * repeat_oof["Huber"]
    repeat_cv_rows.append({
        "Split seed": split_seed,
        "RMSE model": rmse(y, repeat_oof["RMSE"]),
        "Huber model": rmse(y, repeat_oof["Huber"]),
        "Fixed 0.56/0.44 blend": rmse(y, fixed_blend),
    })
repeat_cv_table = pd.DataFrame(repeat_cv_rows)
print("\nFresh split stability check (INR RMSE):")
print(repeat_cv_table.to_string(index=False, formatters={
    col: lambda value: f"{value:,.0f}"
    for col in ["RMSE model", "Huber model", "Fixed 0.56/0.44 blend"]
}))


# Additional validation designs: target-stratified folds, one stratified holdout,
# and City-grouped folds. The group check measures performance on unseen cities.
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split

target_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
stratified_folds = list(
    StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, target_bins)
)
holdout_train, holdout_valid = train_test_split(
    np.arange(len(y)), test_size=0.20, random_state=2026, stratify=target_bins
)
city_groups = train["City"].fillna("Missing").astype(str).to_numpy()
city_folds = list(GroupKFold(n_splits=5).split(X, y, city_groups))

def evaluate_validation_design(name, split_pairs, seed_offset=2026):
    model_oof = {"RMSE": np.zeros(len(y)), "Huber": np.zeros(len(y))}
    validation_indices = []
    for fold_number, (train_idx, valid_idx) in enumerate(split_pairs, start=1):
        validation_indices.extend(valid_idx.tolist())
        for model_name, loss in [
            ("RMSE", "RMSE"),
            ("Huber", "Huber:delta=12000000"),
        ]:
            model = CatBoostRegressor(
                loss_function=loss,
                eval_metric="RMSE",
                iterations=1800,
                learning_rate=0.04,
                depth=6,
                l2_leaf_reg=5,
                random_seed=seed_offset + fold_number,
                thread_count=4,
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(
                X.iloc[train_idx], y.iloc[train_idx],
                cat_features=cat_cols,
                eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
                early_stopping_rounds=120,
                verbose=False,
            )
            model_oof[model_name][valid_idx] = np.maximum(
                model.predict(X.iloc[valid_idx]), 0
            )
            print(f"{name} | fold {fold_number} | {model_name} complete")
    scored_idx = np.asarray(validation_indices, dtype=int)
    blend = 0.55 * model_oof["RMSE"] + 0.45 * model_oof["Huber"]
    y_scored = y.iloc[scored_idx]
    return {
        "Validation": name,
        "RMSE CatBoost": rmse(y_scored, model_oof["RMSE"][scored_idx]),
        "Huber CatBoost": rmse(y_scored, model_oof["Huber"][scored_idx]),
        "Fixed 0.55/0.45 blend": rmse(y_scored, blend[scored_idx]),
    }

alternate_validation_rows = []
alternate_validation_rows.append(
    evaluate_validation_design("Target-stratified 5-fold", stratified_folds)
)
alternate_validation_rows.append(
    evaluate_validation_design(
        "Stratified 80/20 holdout", [(holdout_train, holdout_valid)], seed_offset=2025
    )
)
alternate_validation_rows.append(
    evaluate_validation_design("City GroupKFold", city_folds)
)
alternate_validation_table = pd.DataFrame(alternate_validation_rows)
print("\nAlternate validation checks (INR RMSE):")
print(alternate_validation_table.to_string(index=False, formatters={
    col: lambda value: f"{value:,.0f}"
    for col in ["RMSE CatBoost", "Huber CatBoost", "Fixed 0.55/0.45 blend"]
}))
print("Distinct training cities:", len(np.unique(city_groups)))
test_cities = set(test["City"].fillna("Missing").astype(str))
train_cities = set(city_groups)
print("Cities shared with test:", len(train_cities & test_cities), "of", len(test_cities))
sample_constant = float(sample["Annual Turnover"].median())
print(
    f"Sample constant baseline ({sample_constant:,.0f} INR) train RMSE:",
    f"{rmse(y, np.full(len(y), sample_constant)):,.0f}",
)


## 6. Choose the validated model and fit all training rows

The final pair and blend weight are selected from the primary shuffled five-fold out-of-fold comparison. Repeated shuffled folds, target-stratified folds, a stratified 80/20 holdout, and City GroupKFold are also reported as robustness checks. The City check is a stress test for unseen locations; the other checks better resemble random test sampling. These scores are estimates from labeled training data, not the hidden-test score.


In [ ]:
single_scores = {name: rmse(y, pred) for name, pred in oof_predictions.items()}
best_single = min(single_scores, key=single_scores.get)

best_blend = None
keys = list(oof_predictions.keys())
for i, left in enumerate(keys):
    for right in keys[i + 1:]:
        for weight in np.linspace(0.05, 0.95, 19):
            blended = weight * oof_predictions[left] + (1 - weight) * oof_predictions[right]
            score = rmse(y, blended)
            if best_blend is None or score < best_blend["score"]:
                best_blend = {"left": left, "right": right, "weight": float(weight), "score": score}

if best_blend and best_blend["score"] <= single_scores[best_single] - 50_000:
    selected = [best_blend["left"], best_blend["right"]]
    blend_weight = best_blend["weight"]
    selected_score = best_blend["score"]
    print(f"Selected blend: {blend_weight:.2f} * {selected[0]} + {1-blend_weight:.2f} * {selected[1]}")
else:
    selected = [best_single]
    blend_weight = None
    selected_score = single_scores[best_single]
    print("Selected single model:", best_single)

print(f"Selected out-of-fold RMSE: INR {selected_score:,.0f}")
print("Best single out-of-fold RMSE:", f"INR {single_scores[best_single]:,.0f}")
print("Leaderboard reference from screenshot: approximately INR 11,946,659")
print("Cross-validation and leaderboard scores can differ; no test labels are available here.")

def fit_full_model(name):
    info = candidate_info[name]
    if info["kind"] == "catboost":
        iterations = int(np.median(best_iterations[name]))
        iterations = max(iterations, 50)
        target = np.log1p(y) if info["log_target"] else y
        model = CatBoostRegressor(
            loss_function=info.get("objective", "RMSE"),
            iterations=iterations,
            learning_rate=0.04,
            depth=info["depth"],
            l2_leaf_reg=5,
            random_seed=RANDOM_STATE,
            thread_count=4,
            verbose=False,
        )
        model.fit(X, target, cat_features=cat_cols, verbose=False)
        return model, info["log_target"], iterations
    model = make_extra_trees_pipeline()
    model.fit(X, y)
    return model, False, None

fitted_models = {}
for name in selected:
    model, is_log, iters = fit_full_model(name)
    fitted_models[name] = (model, is_log)
    if iters is not None:
        print(f"Final fit {name}: {iters} iterations")
    else:
        print(f"Final fit {name}: Extra Trees pipeline")

## 7. Generate and check the submission file

In [ ]:
def predict_full(name):
    model, is_log = fitted_models[name]
    pred = model.predict(X_test)
    if is_log:
        pred = np.expm1(pred)
    return np.maximum(np.asarray(pred, dtype=float), 0)

if len(selected) == 1:
    test_predictions = predict_full(selected[0])
else:
    test_predictions = (
        blend_weight * predict_full(selected[0])
        + (1 - blend_weight) * predict_full(selected[1])
    )

submission = pd.DataFrame({
    "Registration Number": test_ids.to_numpy(),
    "Annual Turnover": test_predictions,
})

assert list(submission.columns) == ["Registration Number", "Annual Turnover"]
assert len(submission) == 500
assert submission.isna().sum().sum() == 0
assert (submission["Annual Turnover"] >= 0).all()
assert submission["Registration Number"].equals(test_ids.reset_index(drop=True))
assert submission["Registration Number"].equals(sample["Registration Number"])

output_path = Path("submission.csv")
submission.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
print("Rows:", len(submission), "| columns:", list(submission.columns))
print("Prediction range (INR):", f"{submission['Annual Turnover'].min():,.0f}", "to", f"{submission['Annual Turnover'].max():,.0f}")
print("Prediction mean (INR):", f"{submission['Annual Turnover'].mean():,.0f}")
print("\nSubmission preview:")
print(submission.head().to_string(index=False))
print("\nAll submission checks passed.")

try:
    from google.colab import files
    files.download(str(output_path))
    print("Triggered download of the checked submission CSV.")
except ImportError:
    print("The CSV is saved in the current notebook folder.")

## Final interpretation

Selected model: **0.55 × CatBoost raw depth 6 + 0.45 × CatBoost Huber delta 12m**. Its primary five-fold out-of-fold RMSE was **INR 19,854,618**, compared with **INR 19,924,168** for the best single model. On two fresh shuffled five-fold assignments, the fixed 0.56/0.44 blend averaged **INR 19,882,042** versus **INR 19,981,387** for standard RMSE CatBoost. The additional target-stratified, held-out, and unseen-city blend RMSEs were **INR 20,004,122**, **INR 20,101,712**, and **INR 20,133,535**, respectively. The holdout is scored only on its held-out rows; City GroupKFold is a stress test for unseen locations. Results support a small gain on random splits, but not a guaranteed leaderboard rank.

The leaderboard screenshot showed a best score of about INR 11.95 million, substantially below this notebook’s validation results. Submit the checked CSV in the hackathon module and this executed notebook separately under LLJ_Oct_2026. Do not submit Sample_Solution.csv as predictions.